In [39]:
import pandas as pd
import pandas_ta_classic as ta
import config
from oandapyV20 import API
import oandapyV20.endpoints.instruments as instruments

In [40]:
#Set up OANDA connection
client = API(access_token=config.OANDA_API_KEY)

In [41]:
#define variables
timeframe = "M5"
instrument = "GBP_JPY"

In [42]:
# Get latest Candle Data
def get_Candles(tf):

    params = {
        "granularity": tf,
        "price": "A"
    }

    r = instruments.InstrumentsCandles(instrument=instrument, params=params)
    candles = client.request(r)['candles']
    candles

    #Convert to Panda's DataFrame
    data = []

    for c in candles:
        if c["complete"]:
            data.append({
                "time": c["time"],
                "open": float(c["ask"]["o"]),
                "high": float(c["ask"]["h"]),
                "low": float(c["ask"]["l"]),
                "close": float(c["ask"]["c"])
            })
    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    return df

In [43]:
# Function: Return EMA Indicators & ATR Stop Loss
def get_indicators(df, high_ind, low_ind):

    #EMAs
    df["EMA_Low"] = ta.ema(df["close"], length=low_ind)
    df["EMA_High"] = ta.ema(df["close"], length=high_ind)

    # ATR Stop_Loss
    df["ATR_14"] = ta.atr(df["high"], df["low"], df["close"], length=14)

    return df


In [44]:
# Funtion: EMA Crossover logic
def ema_crossover(df):
    #Take profit ration 1.5%
    tp_ratio = 1.5

    last_candle = df.iloc[-1]
    previous_candle = df.iloc[-2]

    if last_candle['EMA_Low'] > last_candle['EMA_High'] and previous_candle['EMA_Low'] < previous_candle['EMA_High']:
        print("Crossover Buy Signal is generated")
    else:
        print ("Crossover Strategy conditions not met")
    

In [45]:
def run_bot():
    print("Starting TradePI Bot ......")
    price = get_Candles(timeframe)
    price = get_indicators(price, high_ind=8, low_ind=5)
    ema_crossover(price)
    


In [46]:
run_bot()

Starting TradePI Bot ......
Crossover Strategy conditions not met
